# WEEK 2 DAY 4: AFL Player Performance Investigation

# 1. Loading and Importing

In [1]:
#importing libs
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
#loading the datasets
rounds = pd.read_csv("afl_players_round_by_round_stats_raw.csv")
players = pd.read_csv("players_info.csv")
seasonal = pd.read_csv("seasonal_stats.csv")
rounds.head()

,id,team,year,career_game_count,opponent,round,result,jersey_num,kicks,marks,...,marks_inside_50,one_percenters,bounces,goal_assist,percentage_of_game_played,player_id,match_date,fantasy_points,score,margin
0,556392,Hawthorn Hawks,1994,17,Richmond Tigers,21,W,34,5.0,4.0,...,NaN,NaN,NaN,NaN,NaN,45552,1994-08-14,36,NaN,28
1,614897,Geelong Cats,2024,1,St Kilda Saints,1,W,7,5.0,NaN,...,NaN,1.0,NaN,NaN,26.0,44356,2024-03-16,23,NaN,8
2,583553,Essendon Bombers,1999,97,Adelaide Crows,10,W,6,14.0,5.0,...,NaN,3.0,NaN,NaN,NaN,45955,1999-06-04,67,NaN,48
3,590676,Western Bulldogs,1994,36,St Kilda Saints,21,W,35,12.0,10.0,...,NaN,NaN,NaN,NaN,NaN,45656,1994-08-13,81,NaN,45
4,582473,Richmond Tigers,1997,113,Melbourne Demons,10,L,41,4.0,2.0,...,NaN,NaN,NaN,NaN,NaN,45929,1997-05-31,32,NaN,-25


In [4]:
print("Shape of roundbyround:" ,rounds.shape)
print("Shape of players:" ,players.shape)
print("Shape of seasons:" ,seasonal.shape)

Shape of roundbyround: (274089, 36)
Shape of players: (2843, 16)
Shape of seasons: (25481, 54)


In [6]:
rounds.isna().sum()

id                                0
team                              0
year                              0
career_game_count                 0
opponent                          0
round                             0
result                            0
jersey_num                        0
kicks                          1300
marks                          8051
handballs                      4433
disposals                      8453
goals                         74737
behinds                       81093
hit_outs                      96629
tackles                       28682
rebound_50s                   62856
inside_50s                    46574
clearances                    63919
clangers                      49965
free_kicks_for                53842
free_kicks_against            53275
brownlow_votes               110481
contested_possessions         38826
uncontested_possessions       37998
contested_marks               90677
marks_inside_50               91378
one_percenters              

In [7]:
rounds.score.isna().sum()

np.int64(274089)

In [8]:
rounds.isna().mean() * 100

id                             0.000000
team                           0.000000
year                           0.000000
career_game_count              0.000000
opponent                       0.000000
round                          0.000000
result                         0.000000
jersey_num                     0.000000
kicks                          0.474298
marks                          2.937367
handballs                      1.617358
disposals                      3.084035
goals                         27.267420
behinds                       29.586375
hit_outs                      35.254607
tackles                       10.464484
rebound_50s                   22.932697
inside_50s                    16.992291
clearances                    23.320527
clangers                      18.229480
free_kicks_for                19.643984
free_kicks_against            19.437117
brownlow_votes                40.308440
contested_possessions         14.165472
uncontested_possessions       13.863380


In [9]:
print(rounds[rounds['goals'].isna()][['team','disposals','tackles','goals']].head())
print()
print('kicks null %:', rounds['kicks'].isna().mean()*100)
print('fantasy_points null %:', rounds['fantasy_points'].isna().mean()*100)



               team  disposals  tackles  goals
0    Hawthorn Hawks        2.0      2.0    NaN
1      Geelong Cats        NaN      2.0    NaN
2  Essendon Bombers       14.0      NaN    NaN
3  Western Bulldogs       15.0      2.0    NaN
4   Richmond Tigers        3.0      1.0    NaN

kicks null %: 0.47429849428470316
fantasy_points null %: 0.0


# 2. Cleaning

* score column in the round data is 100% empty, so we are going to drop it
* Other round-level stat columns have NULLs where an event just didn't happen so i filled it with 0.
  
kicks, disposals, fantasy_points are basically never NULL, meaning the match was definitely tracked for that player. So when goals is null on a fully tracked row, it's not that the data is missing, it's that the player just didn't kick a goal that game. That's the reasoning for filling with 0 instead of dropping or leaving as nan

* seasonal df has separate rows for regular season and finals for the same player-year. These are combined into one row per player per season so players arent double counted

In [10]:
rounds

,id,team,year,career_game_count,opponent,round,result,jersey_num,kicks,marks,...,marks_inside_50,one_percenters,bounces,goal_assist,percentage_of_game_played,player_id,match_date,fantasy_points,score,margin
0,556392,Hawthorn Hawks,1994,17,Richmond Tigers,21,W,34,5.0,4.0,...,NaN,NaN,NaN,NaN,NaN,45552,1994-08-14,36,NaN,28
1,614897,Geelong Cats,2024,1,St Kilda Saints,1,W,7,5.0,NaN,...,NaN,1.0,NaN,NaN,26.0,44356,2024-03-16,23,NaN,8
2,583553,Essendon Bombers,1999,97,Adelaide Crows,10,W,6,14.0,5.0,...,NaN,3.0,NaN,NaN,NaN,45955,1999-06-04,67,NaN,48
3,590676,Western Bulldogs,1994,36,St Kilda Saints,21,W,35,12.0,10.0,...,NaN,NaN,NaN,NaN,NaN,45656,1994-08-13,81,NaN,45
4,582473,Richmond Tigers,1997,113,Melbourne Demons,10,L,41,4.0,2.0,...,NaN,NaN,NaN,NaN,NaN,45929,1997-05-31,32,NaN,-25
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
274084,18472,Adelaide Crows,2016,120,Sydney Swans,SF,L,33,7.0,4.0,...,0.0,3.0,0.0,0.0,80.0,44961,2016-09-17,51,NaN,-36
274085,25680,Greater Western Sydney Giants,2021,26,Geelong Cats,SF,L,39,6.0,4.0,...,0.0,4.0,0.0,0.0,79.0,44123,2021-09-03,38,NaN,-35
274086,618575,Melbourne Demons,2019,2,Gold Coast Suns,8,W,45,3.0,1.0,...,1.0,5.0,NaN,NaN,77.0,46165,2019-05-11,46,NaN,1
274087,617689,St Kilda Saints,2025,1,Adelaide Crows,1,L,31,4.0,1.0,...,NaN,2.0,NaN,NaN,78.0,46021,2025-03-16,68,NaN,-63


In [11]:
rounds = rounds.drop(columns=['score'])   #dropped scores, all rows are null

count_stat_cols = ['marks','handballs','disposals','goals','behinds','hit_outs','tackles','rebound_50s',
    'inside_50s','clearances','clangers','free_kicks_for','free_kicks_against','brownlow_votes',
    'contested_possessions','uncontested_possessions','contested_marks','marks_inside_50','one_percenters',
    'bounces','goal_assist']
rounds[count_stat_cols] = rounds[count_stat_cols].fillna(0)
rounds['round_num'] = pd.to_numeric(rounds['round'], errors='coerce')

print(rounds['round_num'])

0         21.0
1          1.0
2         10.0
3         21.0
4         10.0
          ... 
274084     NaN
274085     NaN
274086     8.0
274087     1.0
274088     9.0
Name: round_num, Length: 274089, dtype: float64


In [12]:
rounds.isna().sum()

id                               0
team                             0
year                             0
career_game_count                0
opponent                         0
round                            0
result                           0
jersey_num                       0
kicks                         1300
marks                            0
handballs                        0
disposals                        0
goals                            0
behinds                          0
hit_outs                         0
tackles                          0
rebound_50s                      0
inside_50s                       0
clearances                       0
clangers                         0
free_kicks_for                   0
free_kicks_against               0
brownlow_votes                   0
contested_possessions            0
uncontested_possessions          0
contested_marks                  0
marks_inside_50                  0
one_percenters                   0
bounces             